# Segmentation and Multi-Task Evaluation

This notebook evaluates the obfuscation framework across diverse vision tasks:
- **SegFormer** — Semantic segmentation on ADE20K
- **CLIPSeg** — Zero-shot segmentation
- **CLIP** — Zero-shot classification
- **OWL-ViT** — Zero-shot object detection

This demonstrates the framework's task-agnostic nature and plug-and-play compatibility.

In [ ]:
import torch
import logging

logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(name)s - %(levelname)s - %(message)s")
# Suppress noisy HTTP logs
for name in ["httpx", "httpcore", "filelock", "huggingface_hub"]:
    logging.getLogger(name).setLevel(logging.WARNING)

print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## Step 1: Segmentation Experiments

In [ ]:
from vit_obfuscation.config.experiment import ExperimentConfig
from vit_obfuscation.runner.runner import ExperimentRunner

segmentation_configs = {
    "SegFormer (ADE20K)": "../configs/experiments/segformer.yaml",
    "CLIPSeg (Zero-shot)": "../configs/experiments/clipseg.yaml",
}

seg_results = {}
for name, path in segmentation_configs.items():
    print(f"\n{'='*60}")
    print(f"Running {name}...")
    print(f"{'='*60}")
    config = ExperimentConfig.from_yaml(path)
    runner = ExperimentRunner(config)
    results = runner.run()
    seg_results[name] = results
    print(f"{name} complete")

## Step 2: Zero-Shot Experiments

In [ ]:
zero_shot_configs = {
    "CLIP (Zero-shot Classification)": "../configs/experiments/clip_zero_shot.yaml",
    "OWL-ViT (Zero-shot Detection)": "../configs/experiments/owlvit_coco.yaml",
}

zs_results = {}
for name, path in zero_shot_configs.items():
    print(f"\n{'='*60}")
    print(f"Running {name}...")
    print(f"{'='*60}")
    config = ExperimentConfig.from_yaml(path)
    runner = ExperimentRunner(config)
    results = runner.run()
    zs_results[name] = results
    print(f"{name} complete")

## Step 3: Segmentation Visualization

Visualize segmentation predictions with and without obfuscation.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image
from transformers import SegformerForSemanticSegmentation, SegformerImageProcessor
from vit_obfuscation.adapter import create_adapter
from vit_obfuscation.obfuscation import ObfuscationModule

# Load SegFormer model and processor
model_name = "nvidia/segformer-b0-finetuned-ade-512-512"
processor = SegformerImageProcessor.from_pretrained(model_name)
model = SegformerForSemanticSegmentation.from_pretrained(model_name)
model.eval()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

# Load a sample ADE20K image
from datasets import load_dataset
ade20k = load_dataset("scene_parse_150", split="validation", streaming=True)
sample = next(iter(ade20k))
image = sample["image"].convert("RGB")

# Prepare inputs
inputs = processor(images=image, return_tensors="pt").to(device)

# Clean prediction
with torch.no_grad():
    clean_outputs = model(**inputs)
clean_logits = clean_outputs.logits
clean_mask = clean_logits.argmax(dim=1).squeeze().cpu().numpy()
clean_mask_resized = np.array(Image.fromarray(clean_mask.astype(np.uint8)).resize(image.size, Image.NEAREST))

# Obfuscated prediction
adapter = create_adapter(model)
obf_module = ObfuscationModule(adapter)
obf_module.to(device)

with torch.no_grad():
    obf_outputs = obf_module(inputs["pixel_values"])
obf_logits = obf_outputs.logits if hasattr(obf_outputs, "logits") else obf_outputs
obf_mask = obf_logits.argmax(dim=1).squeeze().cpu().numpy()
obf_mask_resized = np.array(Image.fromarray(obf_mask.astype(np.uint8)).resize(image.size, Image.NEAREST))

# Visualize
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

axes[0].imshow(image)
axes[0].set_title("Original Image")
axes[0].axis("off")

axes[1].imshow(clean_mask_resized, cmap="tab20", interpolation="nearest")
axes[1].set_title("Clean Prediction")
axes[1].axis("off")

axes[2].imshow(obf_mask_resized, cmap="tab20", interpolation="nearest")
axes[2].set_title("Obfuscated Prediction")
axes[2].axis("off")

plt.suptitle("SegFormer Segmentation: Clean vs Obfuscated", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

## Step 4: Cross-Task Summary

Comprehensive comparison of all task types showing the framework's versatility.

In [ ]:
import pandas as pd
import json, os, glob

rows = []

# Segmentation results
for name, results in seg_results.items():
    clean = results.get("clean", {})
    obf = results.get("obfuscated", {})
    metric_name = "mIoU" if "miou" in str(clean).lower() else "Score"
    clean_val = clean.get("miou", clean.get("accuracy", clean.get("score", 0)))
    obf_val = obf.get("miou", obf.get("accuracy", obf.get("score", 0)))
    rows.append({"Task": name, "Metric": metric_name, "Clean": f"{clean_val:.4f}", "Obfuscated": f"{obf_val:.4f}"})

# Zero-shot results
for name, results in zs_results.items():
    clean = results.get("clean", {})
    obf = results.get("obfuscated", {})
    clean_val = clean.get("accuracy", clean.get("map", clean.get("score", 0)))
    obf_val = obf.get("accuracy", obf.get("map", obf.get("score", 0)))
    metric_name = "mAP" if "detection" in name.lower() else "Accuracy"
    rows.append({"Task": name, "Metric": metric_name, "Clean": f"{clean_val:.4f}", "Obfuscated": f"{obf_val:.4f}"})

# Load any saved classification/detection results
for result_file in sorted(glob.glob("../outputs/*_results.json")):
    try:
        with open(result_file) as f:
            saved = json.load(f)
        if saved.get("task") in ("classification", "object_detection"):
            clean = saved.get("clean", {})
            obf = saved.get("obfuscated", {})
            clean_val = clean.get("accuracy", clean.get("map", 0))
            obf_val = obf.get("accuracy", obf.get("map", 0))
            metric = "mAP" if "detection" in saved["task"] else "Accuracy"
            rows.append({"Task": saved["experiment"], "Metric": metric, "Clean": f"{clean_val:.4f}", "Obfuscated": f"{obf_val:.4f}"})
    except Exception:
        pass

df = pd.DataFrame(rows)
print("Cross-Task Summary:")
print(df.to_string(index=False))

## Discussion

The results demonstrate that the obfuscation framework preserves task utility across fundamentally
different vision tasks — from pixel-level segmentation to text-guided zero-shot reasoning.
This validates the framework's claim of modular, task-agnostic design:

- **Segmentation**: Dense per-pixel predictions are maintained through obfuscation
- **Zero-shot tasks**: Vision-language models (CLIP, OWL-ViT, CLIPSeg) remain functional
- **No retraining required**: Frozen backbones work directly with the deobfuscation embedding